# FDSI benchmark: results

How much online adaptation improves motor-unit tracking over no adaptation, across the FDSI
benchmark's 100 recordings (5 subjects × 5 contractions × 4 SNR levels), for each way of
tuning it: `sv_loss` alone, the Pareto front of `wh_loss` and `sv_loss` (min-sv member), each
with `sv_loss` averaged or summed over units, and the ground-truth RoA as an upper bound.

This notebook only reads the tables written by `python -m benchmarks.fdsi collect` (and
`import-v10`) under the spec's `outputs_root/tables/`. Set the `FDSI_SPEC` environment
variable to read another spec's tables (e.g. `benchmarks/fdsi/benchmark_smoke.yaml`).

> **Results pending.** This page shows the report's code; its figures and tables are added
> when the release benchmark is run (see the benchmark page's release checklist).

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "benchmarks" / "fdsi").is_dir()
)
sys.path.insert(0, str(REPO_ROOT))  # so that benchmarks.fdsi imports from any working directory

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from adapt_decomp.utils.plots import (
    plot_metric_heatmap,
    plot_phase_bar,
    plot_search_front,
    plot_search_landscape,
    plot_search_parameters,
)
from benchmarks.fdsi import report
from benchmarks.fdsi.spec import load_spec

spec = load_spec(os.environ.get("FDSI_SPEC", "benchmarks/fdsi/benchmark.yaml"))
tables = report.load_tables(spec)
units, calib_units = tables["units"], tables["calibration_units"]
searches, best_configs = tables["searches"], tables["best_configs"].set_index("search")

LABELS = {branch: report.config_label(spec, branch) for branch in spec.branches}
CONFIGS = report.config_order(spec)
FIXED, ADAPTED = report.FIXED_LABEL, CONFIGS[1:]
CONDITIONS, SNR_LEVELS = spec.grid["conditions"], spec.grid["snr_levels"]
TRIANGULAR = [c for c in CONDITIONS if "triangular" in c]
PALETTE = dict(zip(CONFIGS, sns.color_palette("tab10", len(CONFIGS))))
pd.set_option("display.precision", 2)

print(
    f"{spec.name}: {len(spec.recordings())} recordings, {len(CONFIGS)} configs, {len(units)} unit rows"
)

## 1. Run overview

Which tasks ran, from which commit, on which machines and for how long (every output also
has its own `*.meta.yaml` with the full record).

In [ ]:
display(report.provenance_summary(tables["provenance"]))

### Calibration

CBSS on the first 5 s of each recording, keeping the units that match a simulated motor unit
with RoA ≥ 90 %. Per condition and SNR: units per recording, their RoA against the ground
truth, and how many have SIL ≥ 0.9 (all averaged over subjects).

In [ ]:
calib_rec = report.units_per_recording(
    calib_units,
    sil_col="sil_calib",
    roa_col="roa_calib",
    by=["recording", "sub", "condition", "snr"],
).assign(config="Calibration")
calib_units_c = calib_units.assign(config="Calibration")

fig, axs = plt.subplots(1, 3, figsize=(16, 4), layout="constrained")
plot_metric_heatmap(calib_rec, "n_units", ["Calibration"], CONDITIONS, SNR_LEVELS, axs=axs[[0]])
plot_metric_heatmap(
    calib_units_c,
    "roa_calib_pct",
    ["Calibration"],
    CONDITIONS,
    SNR_LEVELS,
    vmin=90,
    vmax=100,
    axs=axs[[1]],
)
plot_metric_heatmap(calib_rec, "n_sil_ge", ["Calibration"], CONDITIONS, SNR_LEVELS, axs=axs[[2]])
for ax, title in zip(
    axs, ["Units per recording", "Calibration RoA (%)", "Units with SIL ≥ 0.9 per recording"]
):
    ax.set_title(title)
plt.show()

## 2. Searches

Each search tuned `wh_learning_rate`, `sv_learning_rate` and `centroid_momentum` on the pool's
recordings, from the end of their calibration window. The chosen trial's parameters are the
config applied to every recording.

In [ ]:
display(
    best_configs.set_index("config")[
        [
            "objectives",
            "selection",
            "sv_loss_reduction",
            "chosen_trial",
            "wh_learning_rate",
            "sv_learning_rate",
            "centroid_momentum",
        ]
    ]
)

### Landscapes

Every trial's pooled RoA against the pooled losses: whether a lower loss means better
tracking, and where the chosen trial sits.

In [ ]:
names = list(best_configs.index)
fig, axs = plt.subplots(
    len(names), 3, figsize=(14, 3.4 * len(names)), layout="constrained", sharey=True, squeeze=False
)
for row, name in zip(axs, names):
    plot_search_landscape(
        report.complete_trials(searches, name),
        best_configs.loc[name, "chosen_trial"],
        title=LABELS[name],
        axs=row,
    )
plt.show()

### Pareto fronts

In [ ]:
pareto = [
    n
    for n in names
    if "values_wh_loss" in report.objective_columns(report.complete_trials(searches, n))
]
if pareto:
    fig, axs = plt.subplots(
        1, len(pareto), figsize=(6 * len(pareto), 4.4), layout="constrained", squeeze=False
    )
    for ax, name in zip(axs[0], pareto):
        trials = report.complete_trials(searches, name)
        plot_search_front(
            trials,
            report.front_numbers(trials),
            best_configs.loc[name, "chosen_trial"],
            title=LABELS[name],
            ax=ax,
        )
    plt.show()

### Convergence

The best objective value reached so far at each trial (for a Pareto search, its `sv_loss`,
which the min-sv selection uses).

In [ ]:
fig, axs = plt.subplots(
    1, len(names), figsize=(3.6 * len(names), 3.2), layout="constrained", squeeze=False
)
for ax, name in zip(axs[0], names):
    trials = report.complete_trials(searches, name)
    columns = report.objective_columns(trials)
    column = "values_sv_loss" if "values_sv_loss" in columns else columns[0]
    curve = report.best_so_far(trials, column)
    ax.scatter(curve["number"], curve["value"], s=10, color="0.6")
    ax.step(curve["number"], curve["best_so_far"], where="post", color=PALETTE[LABELS[name]])
    ax.set(title=LABELS[name], xlabel="Trial", ylabel=column.split("_", 1)[1])
plt.show()

### Hyperparameter scales

Pooled RoA of every trial of every search against each parameter: how sensitive tracking is
to each parameter's scale, whichever objective chose it. The searches share a seed, so their
random start-up trials are the same parameter sets and overlap here.

In [ ]:
plot_search_parameters(report.complete_trials(searches), hue="config")
plt.show()

## 3. Gains over no adaptation

RoA after the calibration window (from 5 s on) of every tracked unit, per config.

In [ ]:
display(
    report.summary_by_config(units)
    .reindex(CONFIGS)
    .rename(columns={"pct_ge_threshold": "% units RoA ≥ 90"})
)

### Units per recording

Mean over recordings of the units tracked, those with SIL ≥ 0.9 and those with RoA ≥ 90 %
after calibration, next to the calibration's own.

In [ ]:
per_rec = report.units_per_recording(units, by=["config", "recording"])
counts = per_rec.groupby("config", sort=False)[
    ["n_units", "n_sil_ge", "pct_sil_ge", "n_roa_ge"]
].mean()
calib_counts = (
    calib_rec[["n_units", "n_sil_ge", "pct_sil_ge", "n_roa_ge"]].mean().rename("Calibration")
)
display(
    pd.concat([calib_counts.to_frame().T, counts.reindex(CONFIGS)]).rename(
        columns={"n_sil_ge": "SIL ≥ 0.9", "pct_sil_ge": "% SIL ≥ 0.9", "n_roa_ge": "RoA ≥ 90 %"}
    )
)

### Paired per-unit gain

Each adapted config against no adaptation, unit by unit (same calibrations, so the same units).

In [ ]:
display(report.paired_summary(units, [(FIXED, config) for config in ADAPTED]))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4), layout="constrained")
sns.boxplot(
    data=units,
    x="config",
    y="roa_after_cal_pct",
    order=CONFIGS,
    hue="config",
    palette=PALETTE,
    legend=False,
    flierprops=dict(marker=".", markersize=3, alpha=0.4),
    ax=ax,
)
ax.set(xlabel="", ylabel="RoA after calibration (%)", ylim=(0, 102))
plt.show()

## 4. Across contractions and SNR

Mean RoA after calibration and the share of units at or above 90 %, per contraction and SNR.

In [ ]:
fig, axs = plt.subplots(2, len(CONFIGS), figsize=(4.6 * len(CONFIGS), 8), layout="constrained")
plot_metric_heatmap(
    units, "roa_after_cal_pct", CONFIGS, CONDITIONS, SNR_LEVELS, vmin=0, vmax=100, axs=axs[0]
)
plot_metric_heatmap(
    units,
    "roa_after_cal_pct",
    CONFIGS,
    CONDITIONS,
    SNR_LEVELS,
    agg="pct_ge_threshold",
    threshold=90,
    vmin=0,
    vmax=100,
    axs=axs[1],
)
plt.show()

Gain over no adaptation, unit by unit, averaged per contraction and SNR.

In [ ]:
delta = report.heatmap_delta(units, FIXED)
limit = delta["delta"].abs().max()
plot_metric_heatmap(
    delta, "delta", ADAPTED, CONDITIONS, SNR_LEVELS, vmin=-limit, vmax=limit, cmap="RdBu", center=0
)
plt.show()

### Pool and held-out recordings

The searches only saw the pool's recordings. Pool conditions on other subjects or SNRs, and
the conditions no search saw, show how well each config generalises.

In [ ]:
display(
    units.pivot_table(
        index="config", columns="split", values="roa_after_cal_pct", aggfunc="mean"
    ).reindex(index=CONFIGS, columns=list(report.SPLITS))
)

## 5. Triangular contractions by phase

Mean RoA in the first isometric hold (including the calibration window), the ramp, and the
last isometric hold, where tracking drifts most without adaptation.

In [ ]:
phases = list(report.PHASES.values())
plot_phase_bar(
    report.phase_long(units), CONFIGS, TRIANGULAR, SNR_LEVELS, phases, {p: p for p in phases}
)
plt.show()

## 6. Objectives and loss aggregation

Unit by unit: averaging against summing `sv_loss` over units, `sv_loss` alone against the
Pareto front, and each unsupervised objective against the ground-truth RoA oracle (the gap
to the upper bound).

In [ ]:
def labels(*branches):
    return [LABELS[b] for b in branches if b in LABELS]


pairs = [
    tuple(labels("sv_mean", "sv_sum")),
    tuple(labels("pareto_mean", "pareto_sum")),
    tuple(labels("sv_mean", "pareto_mean")),
    tuple(labels("sv_sum", "pareto_sum")),
    *[
        (config, LABELS["roa"])
        for config in labels("sv_mean", "sv_sum", "pareto_mean", "pareto_sum")
        if "roa" in LABELS
    ],
]
display(report.paired_summary(units, [pair for pair in pairs if len(pair) == 2]))

## 7. Cost

Processing time per batch against the batch's own duration (a real-time factor below 1
keeps up with the signal), and the compute each stage used.

In [ ]:
display(
    report.cost_summary(tables["recordings"], batch_ms=spec.fixed_config().batch_ms).reindex(
        CONFIGS
    )
)
search_time = (
    tables["provenance"]
    .query("stage == 'search'")
    .assign(config=lambda d: d["id"].map(LABELS), run_time_h=lambda d: d["run_time_s"] / 3600)
)
display(search_time.set_index("config")[["run_time_h", "hostname", "cpu"]])

## 8. v1.0 against this version

The v1.0 results (cached, from `import-v10`) against their counterparts here. The two
versions were calibrated separately, so units are paired by the simulated motor unit they
track (`gt_unit`). v1.0 summed `sv_loss` over units, so its searches pair with the `sum` ones.

In [ ]:
if report.V10_TABLE in tables:
    both = pd.concat([units, tables[report.V10_TABLE]], ignore_index=True)
    pairs = [
        (report.V10_LABELS[v10], LABELS[v11])
        for v10, v11 in report.V10_COUNTERPARTS.items()
        if v11 in LABELS
    ]
    display(report.paired_summary(both, pairs, keys=report.GT_UNIT_KEYS))
    shown = [c for pair in pairs for c in pair]
    display(
        both[both["config"].isin(shown)]
        .pivot_table(
            index="config", columns="condition", values="roa_after_cal_pct", aggfunc="mean"
        )
        .reindex(index=shown, columns=CONDITIONS)
    )
else:
    print(
        "No v1.0 table: run 'python -m benchmarks.fdsi import-v10' with the v1.0 outputs downloaded."
    )

## 9. Reproduce

From the repository root, with the `adapt_decomp` environment and the FDSI data
(`python scripts/download_data.py get fdsi_benchmark-data`):

```sh
python -m benchmarks.fdsi calibrate --all --n-workers 8
python -m benchmarks.fdsi search --all
python -m benchmarks.fdsi apply --all --n-workers 8
python -m benchmarks.fdsi collect
python -m benchmarks.fdsi import-v10   # needs the v1.0 outputs (fdsi_benchmark-outputs)
```

or `bash benchmarks/fdsi/pbs/submit.sh` on a PBS Pro cluster. Every output has a
`*.meta.yaml` next to it with the machine, the git commit and the exact command that
reproduces it.